In [ ]:
import sys
from pathlib import Path

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "wiener_chaos/core/model.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from within the project directory.")
sys.path.insert(0, str(ROOT))
DATA_DIR = ROOT / "Real data"
DATA_PATH = DATA_DIR / "data_pere.csv"

import torch
import numpy as np
from wiener_chaos.pricing.options import get_vegas
import pickle
from math import exp
import pandas as pd
from wiener_chaos.pricing.options import _bs_price, implied_vol_bisection

In [ ]:
# Plotting tools for the supplied option surface.
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 (needed for 3D projection)

In [ ]:
if not DATA_PATH.is_file():
    raise FileNotFoundError(
        "SPX quotes are not distributed. Supply your own data_pere.csv "
        "in Real data/; see README.md for columns and units."
    )
df = pd.read_csv(DATA_PATH)
required = {"strike", "bid", "ask", "mid", "idx_price", "ttm", "rfr", "dvd", "impl_vol"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Input CSV is missing columns: {sorted(missing)}")

In [ ]:
S0 = 100

In [ ]:
df[["strike", "bid", "ask", "mid"]] = S0*df[["strike", "bid", "ask", "mid"]].div(df["idx_price"], axis=0)
df[["ttm", "strike", "bid", "ask", "mid"]] = df[["ttm", "strike", "bid", "ask", "mid"]].round(6)

In [ ]:
df.head()

In [ ]:
df = df.sort_values(by=["ttm", "strike"], ascending=[True, True])

In [ ]:
df["iv_mid"] = df.apply(lambda r: implied_vol_bisection(r["mid"], S0, r["strike"], r["ttm"], r["rfr"], r["dvd"], flag="call",
                                                                 max_iter=1000, tol=1e-12), axis=1)

In [ ]:
df["iv_bid"] = df.apply(lambda r: implied_vol_bisection(r["bid"], S0, r["strike"], r["ttm"], r["rfr"], r["dvd"], flag="call"), axis=1)
df["iv_ask"] = df.apply(lambda r: implied_vol_bisection(r["ask"], S0, r["strike"], r["ttm"], r["rfr"], r["dvd"], flag="call"), axis=1)

In [ ]:
np.abs(np.array(df["iv_mid"] - df["impl_vol"])).max()

In [ ]:
np.abs(np.array(df["iv_bid"] - df["iv_ask"])).max()

In [ ]:
df.head()

In [ ]:
maturities = np.unique(df["ttm"]).tolist()
strikes = [np.array(df.loc[df["ttm"] == T, "strike"]).tolist() for T in maturities]

In [ ]:
maturities

In [ ]:
iv_mid, iv_bid, iv_ask = [], [], []
iv_data = []
for t, T in enumerate(maturities):
    iv_mid_T, iv_bid_T, iv_ask_T = [], [], []
    iv_data_T = []
    for K in strikes[t]:
        iv_mid_T.append(df.loc[(df["ttm"] == T) & (df["strike"] == K), "iv_mid"].item())
        iv_bid_T.append(df.loc[(df["ttm"] == T) & (df["strike"] == K), "iv_bid"].item())
        iv_ask_T.append(df.loc[(df["ttm"] == T) & (df["strike"] == K), "iv_ask"].item())
        iv_data_T.append(df.loc[(df["ttm"] == T) & (df["strike"] == K), "impl_vol"].item())
    iv_mid.append(iv_mid_T)
    iv_bid.append(iv_bid_T)
    iv_ask.append(iv_ask_T)
    iv_data.append(iv_data_T)

In [ ]:
price_mid, price_bid, price_ask = [], [], []
for t, T in enumerate(maturities):
    price_mid_T, price_bid_T, price_ask_T = [], [], []
    for K in strikes[t]:
        price_mid_T.append(df.loc[(df["ttm"] == T) & (df["strike"] == K), "mid"].item())
        price_bid_T.append(df.loc[(df["ttm"] == T) & (df["strike"] == K), "bid"].item())
        price_ask_T.append(df.loc[(df["ttm"] == T) & (df["strike"] == K), "ask"].item())

    price_mid.append(price_mid_T)
    price_bid.append(price_bid_T)
    price_ask.append(price_ask_T)

In [ ]:
r_s = np.unique(df[["ttm", "rfr"]].values, axis=0)[:, -1].tolist()
q_s = np.unique(df[["ttm", "dvd"]].values, axis=0)[:, -1].tolist()

In [ ]:
real_data = {}
real_data["maturities"] = maturities
real_data["strikes"] = strikes
real_data["prices"] = price_mid 
real_data["iv"] = iv_mid 
real_data["iv_bid"] = iv_bid 
real_data["iv_ask"] = iv_ask 

real_data["r_s"] = r_s 
real_data["q_s"] = q_s 
real_data["S0"] = S0 

with (DATA_DIR / "real_data.pkl").open("wb") as f:
    pickle.dump(real_data, f)

In [ ]:
for k in range(len(maturities)):
    if np.diff(np.array(real_data["prices"][k])<0).any() != False:
        print(np.diff(np.array(real_data["prices"][k])<0).any())
    slopes = np.diff(np.array(real_data["prices"][k])) / np.diff(np.array(strikes[k]))
    if (np.diff(slopes) < 0).any() != False:
        print(k)

In [ ]:
k = 5
slopes = np.diff(np.array(real_data["prices"][k])) / np.diff(np.array(strikes[k]))
(np.diff(slopes) < 0).any()

In [ ]:
# Plot the supplied implied-volatility surface.

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa

# Suppose:
# maturities: list like [T0, T1, ..., TJ-1]
# strikes:    list of lists, strikes[j] is a list of strikes for maturity maturities[j]
# ivs:        list of lists, ivs[j][i] is IV at (strikes[j][i], maturities[j])

x, y, z = [], [], []
for T, Ks, IVs in zip(maturities, strikes, iv_data):
    for K, v in zip(Ks, IVs):
        x.append(K)
        y.append(T)
        z.append(v)

x = np.array(x); y = np.array(y); z = np.array(z)

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection="3d")
surf = ax.plot_trisurf(x, y, z, linewidth=0, antialiased=True)
ax.set_xlabel("Strike")
ax.set_ylabel("Maturity")
ax.set_zlabel("Implied Volatility")
ax.set_title("Implied Vol Surface (triangulated)")
fig.colorbar(surf, shrink=0.6, aspect=12).set_label("IV")
plt.tight_layout(); plt.show()


In [ ]:
for k in range(len(maturities)):
    plt.figure()   # <-- this creates a new plot each iteration
    plt.plot(strikes[k], price_mid[k], label="market")
    plt.scatter(strikes[k], price_mid[k])
    
    plt.title(f"T = {maturities[k]}")
    plt.xlabel("Strike")
    plt.ylabel("Implied Volatility")
    plt.legend()
    plt.show()

In [ ]:
for k in range(len(maturities)):
    plt.figure()   # <-- this creates a new plot each iteration
    plt.plot(strikes[k], iv_data[k], label="market")
    plt.scatter(strikes[k], iv_data[k])
    
    plt.title(f"T = {maturities[k]}")
    plt.xlabel("Strike")
    plt.ylabel("Implied Volatility")
    plt.legend()
    plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
for j, T in enumerate(maturities):
    plt.plot(
        strikes[j], iv_data[j],
        marker="o", linewidth=1.8, label=f"T = {T:.2f}"
    )

plt.xlabel("Strike")
plt.ylabel("Implied Volatility")
plt.title("IV Smiles by Maturity")
plt.legend(title="Maturity")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()